## Design a Parking Lot

The key requirements are:
1. parking lot with multiple slot types.
2. Supports Bikes, cars, Auto.
3. Dynamic slot allocation based on vehicle type.
4. Payment processing with multiple methods.
5. Entry ticket issuance & Exit validation. 

Ref: https://github.com/aryan-0077/CWA-LowLevelDesignCode/tree/main/07_LLD_Interview_Problems/04_Design_Parking_Lot

### Getting the Key components

#### Vehicle abstract class

In [ ]:
from abc import ABC, abstractmethod
from enum import Enum, auto

# ─────────────────────────────────────────────
#  Vehicle (Abstract)
# ─────────────────────────────────────────────

class VehicleType(Enum):
    CAR        = auto()
    TRUCK      = auto()
    MOTORCYCLE = auto()
    VAN        = auto()

class Vehicle(ABC):
    def __init__(self, license_plate: str, vehicle_type: VehicleType):
        self.license_plate = license_plate
        self.vehicle_type  = vehicle_type
 
    def __repr__(self) -> str:
        return f"{self.__class__.__name__}({self.license_plate})"
 
 
class Car(Vehicle):
    def __init__(self, license_plate: str):
        super().__init__(license_plate, VehicleType.CAR)
 
 
class Truck(Vehicle):
    def __init__(self, license_plate: str):
        super().__init__(license_plate, VehicleType.TRUCK)
 
 
class Motorcycle(Vehicle):
    def __init__(self, license_plate: str):
        super().__init__(license_plate, VehicleType.MOTORCYCLE)
 
 
class Van(Vehicle):
    def __init__(self, license_plate: str):
        super().__init__(license_plate, VehicleType.VAN)
 

### parking slot

Parking slot represents an individual parking space.

In [ ]:
from typing import Dict, List, Optional

class SlotType(Enum):
    COMPACT    = auto()
    LARGE      = auto()
    MOTORCYCLE = auto()
    HANDICAPPED = auto()

SLOT_VEHICLE_COMPATIBILITY: Dict[SlotType, List[VehicleType]] = {
    SlotType.COMPACT:     [VehicleType.CAR, VehicleType.VAN],
    SlotType.LARGE:       [VehicleType.TRUCK, VehicleType.VAN, VehicleType.CAR],
    SlotType.MOTORCYCLE:  [VehicleType.MOTORCYCLE],
    SlotType.HANDICAPPED: [VehicleType.CAR, VehicleType.VAN],
}

class ParkingSlot:
    def __init__(self, slot_number: str, slot_type: SlotType):
        self.slot_number = slot_number
        self.slot_type   = slot_type
        self.is_free     = True
        self.vehicle: Optional[Vehicle] = None
 
    def can_fit(self, vehicle: Vehicle) -> bool:
        return vehicle.vehicle_type in SLOT_VEHICLE_COMPATIBILITY[self.slot_type]
 
    def assign_vehicle(self, vehicle: Vehicle) -> None:
        # is_free is already set to False by get_free_slot() inside the floor lock.
        # We only set it here if called directly (e.g. in tests).
        self.vehicle = vehicle
        self.is_free = False
 
    def remove_vehicle(self) -> None:
        self.vehicle = None
        self.is_free = True
 
    def __repr__(self) -> str:
        state = "FREE" if self.is_free else f"OCC({self.vehicle.license_plate})"
        return f"Slot[{self.slot_number} {self.slot_type.name} {state}]"

### Parking Lot 

Manages parking slots and vehicle allocations.

Responsible for:
1. Allocating and releasing parking slots.
2. Tracking occuplied and free slots.

In [ ]:
class ParkingLot:
    def __init__(self, parking_slots: List[ParkingSlot]):
        self.parking_slots = parking_slots

### Payment Strategy

Payment class handles different payment methods like credit card, Cash, UPI before exit.

Supports Multiple payment menthods: Credit card, UPI, Cash

In [ ]:
class PaymentStrategy(ABC):
    @abstractmethod
    def process_payment(self, amount: float) -> bool:
        pass

### Parking fee Strategy

Defines a common interface for different parking fee calculation strategies.
Supports multiple procing strategies:
1. Basic Rate
2. Premium Rate

Enables flexibility by letting us switch between strategies dynamically.

In [ ]:
class ParkingFeeStrategy(ABC):
    @abstractmethod
    def calculate_fee(self, vehicle: Vehicle, hours: int) -> float:
        pass

### Design Pattern Strategy Used:

1. Factory pattern for vehicle creation:

    -> Allows easy extensions for new vehicle types.
    -> Ensure consistent object creation.

2. Strategy Pattern for payment and Parkingfares:
    -> Enables flecible method and dynamic fare calculation
    -> Easily extenable for future payment integration.

3. Singleton pattern for parking lot management:
    -> Ensure only one instance of parking lot exists at a time.

4. Observer Pattern for Exit Notification:
    -> Notifies the system when the vehicle exits.
    -> Can be extended for alerts or logging future enhancements.